# NBA Champion Predictor — Modeling

Model bake-off, hyperparameter grid search, and final predictions.
Assumes `exploration.ipynb` has been run and `output/all_seasons.csv` exists,
OR that you run the collection step in cell 1b below.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor

from nba_predictor.config import (
    RANDOM_SEED, TRAIN_CUTOFF, GRID_SEARCH_TRAIN_CUTOFF,
    CATEGORICAL_COLS, META_COLS,
)
from nba_predictor.features.selection import select_features
from nba_predictor.features.preprocessing import build_preprocessor, get_header_names
from nba_predictor.models.bakeoff import run_bakeoff
from nba_predictor.models.grid_search import run_grid_search, results_to_dataframe
from nba_predictor.models.evaluate import predict_per_season
from nba_predictor.models.predict import save_model

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 1. Load Data

In [ ]:
# Option A: load pre-scraped CSV
# all_seasons_df = pd.read_csv('../output/all_seasons.csv')

# Option B: scrape live (uncomment below)
from nba_predictor.scraping import collect_all_seasons
seasons_df = collect_all_seasons()
selected_features = select_features(seasons_df)
all_seasons_df = seasons_df.loc[:, selected_features]

## 2. Bake-off: Cross-Validated Model Comparison

In [ ]:
all_seasons_df.sort_values(by=['season'], ascending=False, inplace=True)

train_df = all_seasons_df[all_seasons_df['season'] <= TRAIN_CUTOFF]
test_df = all_seasons_df[all_seasons_df['season'] > TRAIN_CUTOFF]

X_train = train_df.drop(['Team', 'season', 'Champion_Share_Score'], axis=1)
y_train = train_df['Champion_Share_Score']

preprocessor = build_preprocessor(X_train.columns.tolist())
X_train_arr = preprocessor.fit_transform(X_train)

groups = train_df['season'].values

bakeoff_results = run_bakeoff(X_train_arr, y_train, groups)

## 3. Grid Search Split

In [ ]:
all_seasons_df.sort_values(by=['season'], ascending=False, inplace=True)

seasons_till_cutoff = all_seasons_df[all_seasons_df['season'] <= TRAIN_CUTOFF]
seasons_holdout = all_seasons_df[all_seasons_df['season'] > TRAIN_CUTOFF]

df_train = seasons_till_cutoff[seasons_till_cutoff['season'] <= GRID_SEARCH_TRAIN_CUTOFF].copy()
df_test = seasons_till_cutoff[seasons_till_cutoff['season'] > GRID_SEARCH_TRAIN_CUTOFF].copy()
holdout = seasons_holdout.copy()

df_train_extra = df_train[META_COLS].copy()
df_test_extra = df_test[META_COLS].copy()
holdout_extra = holdout[META_COLS].copy()

train_labels = df_train_extra['Champion_Share_Score'].values
test_labels = df_test_extra['Champion_Share_Score'].values

for col in META_COLS:
    df_train = df_train.drop(columns=[col])
    df_test = df_test.drop(columns=[col])
    holdout = holdout.drop(columns=[col])

In [ ]:
preprocessor2 = build_preprocessor(df_train.columns.tolist())

train_features = preprocessor2.fit_transform(df_train)
test_features = preprocessor2.transform(df_test)
holdout_features = preprocessor2.transform(holdout)

header_names = get_header_names(preprocessor2)

In [ ]:
df_test_extra.reset_index(drop=True, inplace=True)
df_test_combined = pd.concat(
    [pd.DataFrame(data=test_features, columns=header_names), df_test_extra], axis=1
)

holdout_extra.reset_index(drop=True, inplace=True)
holdout_combined = pd.concat(
    [pd.DataFrame(data=holdout_features, columns=header_names), holdout_extra], axis=1
)

## 4. GradientBoosting Grid Search

In [ ]:
gb_clf = GradientBoostingRegressor(
    random_state=RANDOM_SEED, criterion='friedman_mse',
    min_samples_split=5, min_samples_leaf=5, max_features=3,
)

gb_results = run_grid_search(
    gb_clf,
    param_grid={
        'max_depth': list(np.arange(3, 11, 1)),
        'n_estimators': [10, 20, 50, 100],
        'min_samples_split': [5, 7, 9],
        'min_samples_leaf': list(np.arange(3, 10, 1)),
        'max_features': list(np.arange(3, 15, 1)),
    },
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    df_test=df_test_combined,
)

all_results = gb_results.copy()

gb_df = results_to_dataframe(gb_results)
gb_df[gb_df['clf'] == 'GradientBoostingRegressor'].head(10)

## 5. GradientBoosting — Best Model Predictions

In [ ]:
gb_model = GradientBoostingRegressor(
    random_state=RANDOM_SEED, n_estimators=10, criterion='friedman_mse',
    max_depth=5, min_samples_split=7, min_samples_leaf=7, max_features=6,
)
gb_model.fit(train_features, train_labels)

gb_predictions = predict_per_season(
    gb_model, holdout_combined, header_names, csv_path='../output/champ_predict_gb.csv'
)

In [ ]:
top_features = pd.Series(gb_clf.feature_importances_, index=header_names).sort_values()
top_features.plot(kind='barh', figsize=(15, 10), title='Top Features (GB)')
plt.show()

## 6. RandomForest & XGBoost Grid Searches

In [ ]:
rf_clf = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-3)
rf_results = run_grid_search(
    rf_clf,
    param_grid={
        'max_depth': list(np.arange(3, 16, 1)),
        'n_estimators': list(np.arange(5, 20, 1)),
    },
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    df_test=df_test_combined,
)

all_results += rf_results

In [ ]:
xgb_clf = XGBRegressor(random_state=RANDOM_SEED, n_jobs=-3, verbosity=0)
xgb_results = run_grid_search(
    xgb_clf,
    param_grid={
        'max_depth': list(np.arange(3, 11, 1)),
        'n_estimators': [10, 20, 50, 100],
        'learning_rate': [0.1, 0.2, 0.3],
        'subsample': list(np.arange(0.4, 1.01, 0.1)),
        'colsample_bytree': list(np.arange(0.4, 1.01, 0.1)),
    },
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    df_test=df_test_combined,
)

all_results += xgb_results

In [ ]:
all_df = results_to_dataframe(all_results)

print('Top RandomForest configs:')
display(all_df[all_df['clf'] == 'RandomForestRegressor'].head(10))

print('\nTop XGBoost configs:')
pd.set_option('display.max_colwidth', None)
display(all_df[all_df['clf'] == 'XGBRegressor'].head(10))

## 7. RandomForest — Best Model Predictions

In [ ]:
rf_model = RandomForestRegressor(
    random_state=RANDOM_SEED, n_jobs=-3, max_depth=10, n_estimators=20,
)
rf_model.fit(train_features, train_labels)

rf_predictions = predict_per_season(
    rf_model, holdout_combined.copy(), header_names,
    csv_path='../output/champ_predict_rf.csv',
)

In [ ]:
top_features = pd.Series(rf_model.feature_importances_, index=header_names).sort_values()
top_features.plot(kind='barh', figsize=(15, 10), title='Top Features (RF)')
plt.show()

## 8. XGBoost — Best Model Predictions

In [ ]:
xgb_model = XGBRegressor(
    random_state=RANDOM_SEED, n_jobs=-3, verbosity=0,
    max_depth=3, n_estimators=20, learning_rate=0.2,
    subsample=0.9, colsample_bytree=0.6,
)
xgb_model.fit(train_features, train_labels)

xgb_predictions = predict_per_season(
    xgb_model, holdout_combined.copy(), header_names,
    csv_path='../output/champ_predict_xgb.csv',
)

In [ ]:
top_features = pd.Series(xgb_model.feature_importances_, index=header_names).sort_values()
top_features.plot(kind='barh', figsize=(15, 10), title='Top Features (XGB)')
plt.show()

## 9. Save Best Model

In [ ]:
save_model(rf_model, '../saved_models/rf_best.joblib')